Imports

In [24]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
Defining Models

# linear baseline, nonlinear single tree, ensemble trees, kernel method, boosted trees
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42),
}

In [ ]:
5-Fold Cross Validation

# stratified keeps class ratios consistent across folds (important because Enrolled is only ~18%)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# macro = compute metric per class then average equally, so minority classes aren't ignored
scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

cv_results = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False)
    cv_results.append({
        "Model": name,
        "Accuracy": str(scores['test_accuracy'].mean()) + " +/- " + str(scores['test_accuracy'].std()),
        "Precision": str(scores['test_precision_macro'].mean()) + " +/- " + str(scores['test_precision_macro'].std()),
        "Recall": str(scores['test_recall_macro'].mean()) + " +/- " + str(scores['test_recall_macro'].std()),
        "F1 macro": str(scores['test_f1_macro'].mean()) + " +/- " + str(scores['test_f1_macro'].std()),
    })
    print("done:", name)

cv_df = pd.DataFrame(cv_results)
display(cv_df)

In [ ]:
Logistic Regression - Tuning Hyperparamters

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["saga"],
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    lr_param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
)
lr_grid.fit(X_train, y_train)

print("Best params:", lr_grid.best_params_)
print("Best CV F1:", lr_grid.best_score_)

lr_top5 = (
    pd.DataFrame(lr_grid.cv_results_)
    [["param_C", "param_penalty", "mean_test_score", "std_test_score"]]
    .sort_values("mean_test_score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
lr_top5.columns = ["C", "Penalty", "Mean F1", "Std F1"]
display(lr_top5)

In [ ]:
Hyperparameter Tunings - SVM 

In [20]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1],
}

svm_grid = GridSearchCV(
    SVC(kernel="rbf", random_state=42),
    svm_param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
)
svm_grid.fit(X_train, y_train)

print("Best params:", svm_grid.best_params_)
print("Best CV F1:", svm_grid.best_score_)

svm_top5 = (
    pd.DataFrame(svm_grid.cv_results_)
    [["param_C", "param_gamma", "mean_test_score", "std_test_score"]]
    .sort_values("mean_test_score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
svm_top5.columns = ["C", "Gamma", "Mean F1", "Std F1"]
display(svm_top5)

Best params: {'C': 10, 'gamma': 0.01}
Best CV F1: 0.6821291320290029


,C,Gamma,Mean F1,Std F1
0,10.0,0.01,0.682129,0.025208
1,10.0,scale,0.674944,0.019151
2,1.0,0.01,0.674700,0.008176
3,10.0,auto,0.672852,0.019489
4,1.0,scale,0.671069,0.012665


In [ ]:
Final CV Comparisons

# replace baseline LR and SVM with their tuned versions, keep the rest as-is
final_models = {
    "Logistic Regression (tuned)": lr_grid.best_estimator_,
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM RBF (tuned)": svm_grid.best_estimator_,
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42),
}

final_cv = []
for name, model in final_models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False)
    final_cv.append({
        "Model": name,
        "Accuracy": str(scores['test_accuracy'].mean()) + " +/- " + str(scores['test_accuracy'].std()),
        "Precision": str(scores['test_precision_macro'].mean()) + " +/- " + str(scores['test_precision_macro'].std()),
        "Recall": str(scores['test_recall_macro'].mean()) + " +/- " + str(scores['test_recall_macro'].std()),
        "F1 macro": str(scores['test_f1_macro'].mean()) + " +/- " + str(scores['test_f1_macro'].std()),
    })

display(pd.DataFrame(final_cv))

In [ ]:
Test Set Evaluation

# only touch test set after all model selection is done
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred, average="macro"))
    print(classification_report(y_test, y_pred, target_names=["Dropout", "Enrolled", "Graduate"]))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Dropout", "Enrolled", "Graduate"],
        cmap="Blues", ax=ax
    )
    ax.set_title(name)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
Best Parameters

In [23]:
print("Logistic Regression:", lr_grid.best_params_)
print("SVM RBF:", svm_grid.best_params_)
print("Decision Tree, Random Forest, XGBoost: default params")

Logistic Regression: {'C': 1, 'penalty': 'l1', 'solver': 'saga'}
SVM RBF: {'C': 10, 'gamma': 0.01}
Decision Tree, Random Forest, XGBoost: default params
